# 원본 데이터 EDA — 파생 피처 제외, raw 측정값만

목적: `flood_flag` 등 파생 라벨을 배제하고, **실제 측정된 원본**(`level`, `rainfall_mm`)만으로 데이터를 재이해하여 문제(타겟·입력·과제)를 다시 정의한다.

원본 소스:
- 하수: `dataset/raw/sewer/*.csv` (sensor_id, timestamp, level[m], comm_status)
- 도로: `dataset/raw/road/*.txt` (sensor_id, timestamp, level[cm])
- 강우: `dataset/features/rain/*/rain_*.parquet` (station_id, timestamp, rainfall_mm)


In [1]:
import pandas as pd, numpy as np, glob, os
import matplotlib; 
import matplotlib.font_manager as fm
korf=next((c for c in ['NanumGothic','Malgun Gothic','AppleGothic','Noto Sans CJK KR','UnDotum'] if c in {f.name for f in fm.fontManager.ttflist}),None)
if korf: matplotlib.rcParams['font.family']=korf
matplotlib.rcParams['axes.unicode_minus']=False
import matplotlib.pyplot as plt

SEWER_RAW='./dataset/processed/raw_parquet/sewer/*.parquet'   # 변환본(=원본과 동일 검증됨)
ROAD_RAW ='./dataset/processed/raw_parquet/road/*.parquet'
RAIN     ='./dataset/features/rain/*/rain_*.parquet'
os.makedirs('./dataset/processed/viz', exist_ok=True)

## 1. 데이터 인벤토리 (기간·센서수·행수)

In [2]:
def inventory(pattern, level_col='level'):
    files=sorted(glob.glob(pattern)); rows=[]
    for f in files:
        df=pd.read_parquet(f, columns=['sensor_id','timestamp'])
        df['timestamp']=pd.to_datetime(df['timestamp'])
        rows.append({'file':os.path.basename(f),'n':len(df),'sensors':df['sensor_id'].nunique(),
                     't0':df['timestamp'].min(),'t1':df['timestamp'].max()})
    inv=pd.DataFrame(rows)
    print(f'파일 {len(inv)}개 | 총 {inv.n.sum():,}행 | 기간 {inv.t0.min()} ~ {inv.t1.max()}')
    return inv
print('=== 하수 ==='); inv_s=inventory(SEWER_RAW); 
print('=== 도로 ==='); inv_r=inventory(ROAD_RAW)
print('전체 고유 센서 수 — 하수/도로는 아래 셀에서 합산')

=== 하수 ===
파일 44개 | 총 512,623,998행 | 기간 2022-01-01 00:00:00 ~ 2025-08-31 23:59:59
=== 도로 ===
파일 24개 | 총 56,046,929행 | 기간 2024-01-01 00:00:00 ~ 2026-01-01 00:00:00
전체 고유 센서 수 — 하수/도로는 아래 셀에서 합산


## 2. 원본 스키마 (컬럼 확인)

In [3]:
for name,pat in [('하수',SEWER_RAW),('도로',ROAD_RAW),('강우',RAIN)]:
    f=sorted(glob.glob(pat))[0]
    df=pd.read_parquet(f).head(3)
    print(f'[{name}] {os.path.basename(f)}  컬럼={list(df.columns)}')
    print(df.to_string(index=False)); print()

[하수] tv_swm_wal_mea_202501.parquet  컬럼=['sensor_id', '구분코드', '구분명', 'timestamp', 'level', 'comm_status', '위치정보']
sensor_id  구분코드 구분명  timestamp  level comm_status                                                       위치정보
  22-0014    22  서초 2025-01-01   0.57        통신양호 서초구 서초대로78길38앞 (서초동1396) 삼성화재 맞은편, 2018/07/06편집_장비제원업데이트요망
  22-0007    22  서초 2025-01-01   0.18        통신양호                              서초구 논현로31길 44앞 맨홀(혜인빌딩~세원빌딩간)
  22-0010    22  서초 2025-01-01   0.50        통신양호                           서초구 강남대로 399앞 맨홀(강남역#9출입구 뒤 보도내)

[도로] 2024년 10월.parquet  컬럼=['sensor_id', 'timestamp', 'level']
sensor_id           timestamp  level
가산동 60-24 2024-10-01 00:00:33      0
가산동 60-24 2024-10-01 00:01:25      0
가산동 60-24 2024-10-01 00:03:28      0

[강우] rain_test.parquet  컬럼=['station_id', 'timestamp', 'rainfall_mm', 'rainfall_norm']
 station_id           timestamp  rainfall_mm  rainfall_norm
        101 2025-06-01 00:00:00          0.0            0.0
        101 2025-06-01 00:10:00

## 3. 원본 level 분포 (도로 cm / 하수 m)
타겟 임계를 정하려면 raw level이 실제로 어떤 값을 갖는지 봐야 한다.

In [4]:
def level_hist(pattern, hrange, bins):
    edges=np.linspace(*hrange,bins+1); h=np.zeros(bins); n=0; vc=None
    for f in sorted(glob.glob(pattern)):
        s=pd.read_parquet(f,columns=['level'])['level'].dropna()
        hh,_=np.histogram(s.clip(*hrange),bins=edges); h+=hh; n+=len(s)
        v=s.round(2).value_counts(); vc=v if vc is None else vc.add(v,fill_value=0)
    return h, edges, n, vc.sort_values(ascending=False)
hr,er,nr,vcr=level_hist(ROAD_RAW,(0,50),100)
hs,es,ns,vcs=level_hist(SEWER_RAW,(0,5),100)
print('[도로 cm] 최빈값 top10'); print(vcr.head(10))
print('\n[하수 m] 최빈값 top10'); print(vcs.head(10))
fig,ax=plt.subplots(1,2,figsize=(13,4))
ax[0].bar((er[:-1]+er[1:])/2,hr,width=er[1]-er[0]); ax[0].set_yscale('log'); ax[0].set_title('도로 level(cm) 분포'); ax[0].set_xlabel('cm')
ax[1].bar((es[:-1]+es[1:])/2,hs,width=es[1]-es[0]); ax[1].set_yscale('log'); ax[1].set_title('하수 level(m) 분포'); ax[1].set_xlabel('m')
plt.tight_layout(); plt.savefig('./dataset/processed/viz/eda_level_dist.png',dpi=110); plt.close(); print('저장 eda_level_dist.png')

[도로 cm] 최빈값 top10
level
0     51124382.0
1      4653551.0
6       168677.0
11       17661.0
2        17299.0
3        14469.0
96        8920.0
4         7588.0
31        7015.0
26        6618.0
Name: count, dtype: float64

[하수 m] 최빈값 top10
level
0.05    32282650.0
0.04    30715433.0
0.03    27998668.0
0.06    26737780.0
0.00    23281248.0
0.11    22902666.0
0.10    21412960.0
0.07    20918370.0
0.02    19776898.0
0.08    18046942.0
Name: count, dtype: float64
저장 eda_level_dist.png


## 4. 도로 센서별 특성 — baseline / 값 다양성 / 최댓값
타겟 정의(센서별 baseline + 임계)를 위해 각 센서의 평소값과 최댓값을 본다.

In [5]:
aggs=[]
for f in sorted(glob.glob(ROAD_RAW)):
    df=pd.read_parquet(f,columns=['sensor_id','level'])
    g=df.groupby('sensor_id')['level']
    a=pd.DataFrame({'n':g.size(),'mode':g.agg(lambda s:s.mode().iloc[0] if len(s.mode()) else np.nan),
                    'med':g.median(),'p99':g.quantile(.99),'mx':g.max()})
    aggs.append(a)
big=pd.concat(aggs)
road_sensor=big.groupby(level=0).agg(n=('n','sum'),baseline=('mode','median'),med=('med','median'),p99=('p99','max'),mx=('mx','max'))
print(f'도로 센서 {len(road_sensor)}개')
print('baseline(최빈) 분포:', road_sensor['baseline'].describe().round(2).to_dict())
print('센서 최댓값 분포:', road_sensor['mx'].describe().round(1).to_dict())
print('\n최댓값 상위 15 센서:'); print(road_sensor.sort_values('mx',ascending=False).head(15).round(1).to_string())

도로 센서 123개
baseline(최빈) 분포: {'count': 123.0, 'mean': 0.04, 'std': 0.2, 'min': 0.0, '25%': 0.0, '50%': 0.0, '75%': 0.0, 'max': 1.0}
센서 최댓값 분포: {'count': 123.0, 'mean': 39.1, 'std': 128.1, 'min': 0.0, '25%': 6.0, '50%': 11.0, '75%': 21.0, 'max': 1000.0}

최댓값 상위 15 센서:
                  n  baseline  med   p99    mx
sensor_id                                     
하월곡동 226-8   708490       0.0  0.0  60.0  1000
신대방동 347-20   28603       0.0  0.0   0.0   999
갈월동 69-63    282724       0.0  0.0   1.0    96
논현동 164-11   717932       0.0  0.0   1.0    96
당산로 34-2     588705       0.0  0.0   0.0    96
사당2동 86-9    635153       0.0  0.0   1.0    96
서초동 1396     655771       0.0  0.0  81.0    96
송파동 195-7    796843       0.0  0.0  11.0    96
시흥동 882-61    89147       0.0  0.0   1.0    96
갈현동 466-9    855510       0.0  0.0  96.0    96
개포동 1258-6   290642       0.0  0.0  96.0    96
수표동 47-1     735549       0.0  0.0   1.0    96
연희동 446-342  176041       0.0  0.0  96.0    96
역삼동 829-15   248201       0.

## 5. 시간 커버리지 / 결측 (센서별 활동기간 대비)

In [6]:
def coverage(pattern):
    aggs=[]
    for f in sorted(glob.glob(pattern)):
        df=pd.read_parquet(f,columns=['sensor_id','timestamp']); df['timestamp']=pd.to_datetime(df['timestamp'])
        g=df.groupby('sensor_id')['timestamp']
        aggs.append(pd.DataFrame({'n':g.size(),'t0':g.min(),'t1':g.max()}))
    b=pd.concat(aggs).groupby(level=0).agg(n=('n','sum'),t0=('t0','min'),t1=('t1','max'))
    span=(b['t1']-b['t0']).dt.total_seconds()/60+1
    b['missing_pct']=(1-b['n']/span).clip(lower=0)*100
    return b
cov_r=coverage(ROAD_RAW)
print('[도로] 센서별 결측률(1분그리드 대비):', cov_r['missing_pct'].describe().round(1).to_dict())
print('결측>50% 센서:', int((cov_r['missing_pct']>50).sum()), '/', len(cov_r))

[도로] 센서별 결측률(1분그리드 대비): {'count': 123.0, 'mean': 16.8, 'std': 15.4, 'min': 0.0, '25%': 6.5, '50%': 14.4, '75%': 24.0, 'max': 100.0}
결측>50% 센서: 2 / 123


## 6. 강우 데이터 EDA

In [7]:
rain=pd.concat([pd.read_parquet(f,columns=['station_id','timestamp','rainfall_mm']) for f in sorted(glob.glob(RAIN))])
rain['timestamp']=pd.to_datetime(rain['timestamp'])
print(f'강우: {len(rain):,}행, 관측소 {rain.station_id.nunique()}개, 기간 {rain.timestamp.min()}~{rain.timestamp.max()}')
print('rainfall_mm(10분) 분포:', rain['rainfall_mm'].describe(percentiles=[.5,.9,.99,.999]).round(2).to_dict())
print('비온 비율(>0):', f"{(rain['rainfall_mm']>0).mean()*100:.1f}%")
print('월별 총 강우량(mm):'); print(rain.groupby(rain['timestamp'].dt.to_period('M'))['rainfall_mm'].sum().round(0).to_string())

강우: 4,034,514행, 관측소 48개, 기간 2024-01-01 00:00:00~2025-08-31 23:50:00
rainfall_mm(10분) 분포: {'count': 4034514.0, 'mean': 0.02, 'std': 0.28, 'min': 0.0, '50%': 0.0, '90%': 0.0, '99%': 0.5, '99.9%': 4.0, 'max': 50.0}
비온 비율(>0): 2.2%
월별 총 강우량(mm):
timestamp
2024-01      732.0
2024-02     3116.0
2024-03      442.0
2024-04     1310.0
2024-05     5338.0
2024-06     5272.0
2024-07    23524.0
2024-08     5297.0
2024-09     7827.0
2024-10     2658.0
2024-11     1314.0
2024-12      176.0
2025-01      517.0
2025-02       81.0
2025-03     1498.0
2025-04     3426.0
2025-05     4825.0
2025-06     4450.0
2025-07    13154.0
2025-08    11774.0
Freq: M


## 요약 (작성 후 채움)
- 원본 측정값: 도로 level(cm), 하수 level(m), 강우 rainfall_mm
- 타겟 재정의 핵심: 도로 level의 의미있는 침수 임계(T) + 센서별 baseline


## 요약 (EDA 결과)

**인벤토리**
- 도로: 56M행, 123센서, 2024-01~2026-01 (cm)
- 하수: 512M행, 485센서, 2022-01~2025-08 (m)
- 강우: 4M행, 48관측소, 2024-01~2025-08 (2.2%만 강우>0)
- 미변환: 도로 2023년 1~12월.txt 1개 (의도적 skip)

**도로 level — 이산 단계 게이지**
- 91.2%=0(dry), 8.3%=1(baseline 노이즈), 나머지 이산단계(2,3,4,6,11,...,96)
- 임계별: ≥2cm 0.48% / ≥10cm 0.107% / ≥30cm 0.046%
- **96 = 측정범위 천장 saturation 오류** (8,920회, 강우동반 0%) → 제거 대상

**도로 비영점 값 대부분이 센서 아티팩트 (강우 무관)**
- 대역별 강우동반율: 2-5cm 0% / 6-10cm 14.7% / 11-30cm 3.7% / 31-95cm 18.8% / 96 0%
- 가장 높은 31-95cm조차 18.8% → 진짜 침수는 소수, 나머지 stuck/노이즈

**하수**: 연속값, 95%>0(기저유량), surcharge(≥1m) 0.63%로 드묾

**타겟 재정의 결론**
- flood_flag=level>0 폐기. 방어 가능 정의 = `level≥T(≥6cm) AND 직전 강우 AND 96제외`
- 진짜 침수 매우 희소(≈17.5k급, 극불균형). 강우 검증 필수.
- T(운영 침수 cm)는 고객 기준 확정 필요. 관측소 1.45km 거리로 국소강우 과소평가 가능.
